# 수출 유망국 대시보드 — EDA

| # | 항목 | 확인하려는 것 |
|---|---|---|
| 1 | 준비 | 데이터 로드, 결측·단위 확인, 스케일 변환 |
| 2 | 박스플롯 | 이상치가 어디에 얼마나 있나 |
| 3 | 상관계수 히트맵 · 가설검정 | 지표끼리 어떻게 얽혀 있나, 그 관계가 통계적으로 유의한가 |
| 4 | 산점도 | 그 관계가 실제로 어떤 모양인가 |
| 5 | 다중공선성 | 중복된 지표는 무엇인가 |

---
## 1. 준비

DB 접속 정보는 코드에 적지 않고 환경변수에서 읽음

프로젝트 루트에 `.env` 파일을 만들거나 환경변수를 직접 설정 (`.env`는 커밋하지 않음)

| 변수 | 내용 |
|---|---|
| `DB_USER` / `DB_PASSWORD` | 계정 |
| `DB_HOST` / `DB_PORT` / `DB_NAME` | 접속 대상 (PORT 기본 3306) |
| `DB_SSL_CA` | RDS 인증서 경로 (기본 `./global-bundle.pem`) |

In [ ]:
# ── 표준 라이브러리 ──────────────────────────────────────────
import os                      # 환경변수(DB 접속 정보) 읽기
from pathlib import Path       # 파일 존재 여부 확인 (.env, 인증서)

# ── 분석·시각화 라이브러리 ───────────────────────────────────
import matplotlib.pyplot as plt
from matplotlib import font_manager   # 설치된 폰트 목록 조회용
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats                                 # 가설검정 (Shapiro-Wilk, Pearson, Spearman)
from statsmodels.stats.multitest import multipletests   # 다중비교 보정 (FDR)
from sqlalchemy import create_engine  # DB 연결 엔진 생성
from sqlalchemy.engine import URL     # 접속 URL을 안전하게 조립 (비밀번호 특수문자 자동 처리)

# seaborn 기본 테마. rcParams를 덮어쓰므로 반드시 폰트 설정보다 "먼저" 호출해야 함
sns.set_theme(style='whitegrid')

# 설치된 한글 폰트 중 첫 번째 것을 사용 (없으면 경고)
# - Noto Sans KR: 팀 공통 / Malgun Gothic: Windows 기본 / NanumGothic: Linux / AppleGothic: macOS
installed = {f.name for f in font_manager.fontManager.ttflist}   # 이 PC에 설치된 폰트 이름 집합
font = next((f for f in ['Noto Sans KR', 'Malgun Gothic', 'NanumGothic', 'AppleGothic']
             if f in installed), None)                           # 후보를 앞에서부터 보며 처음 발견된 것 선택
if font:
    plt.rcParams['font.family'] = font
else:
    print('한글 폰트를 찾지 못함 → 그래프의 한글이 깨질 수 있음')
plt.rcParams['axes.unicode_minus'] = False   # 한글 폰트에서 마이너스(-) 기호가 네모로 깨지는 것 방지

print(plt.rcParams['font.family'])           # 실제로 적용된 폰트 확인

In [ ]:
# ── 1) .env 파일 읽기 ────────────────────────────────────────
# .env 파일이 있으면 읽어서 환경변수로 등록 (이미 설정된 값은 유지)
# 형식: KEY=VALUE 한 줄에 하나. '#'으로 시작하는 줄과 빈 줄은 무시
# 프로젝트 루트 찾기: 현재 폴더에서 위로 올라가며 .env 가 있는 폴더를 찾음
# (노트북이 하위 폴더 안에 있어도 루트의 .env 와 인증서를 읽기 위함)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()), Path.cwd())
env_file = ROOT / '.env'
if env_file.exists():
    for line in env_file.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)   # 첫 번째 '='에서만 나눔 (값 안에 '='가 있어도 안전)
            # setdefault: 이미 같은 이름의 환경변수가 있으면 덮어쓰지 않음. 값 양쪽의 따옴표는 제거
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

# ── 2) 필수 값 확인 ──────────────────────────────────────────
# 빠진 값이 있으면 접속을 시도하기 전에 무엇이 없는지 알려주고 중단
missing = [k for k in ['DB_USER', 'DB_PASSWORD', 'DB_HOST', 'DB_NAME'] if not os.environ.get(k)]
if missing:
    raise RuntimeError(f'환경변수 없음: {", ".join(missing)} (.env 파일 확인)')

# RDS는 SSL 접속이므로 AWS 인증서(global-bundle.pem)가 필요. 경로를 따로 지정하지 않으면 현재 폴더에서 찾음
ssl_ca = Path(os.environ.get('DB_SSL_CA', './global-bundle.pem'))
if not ssl_ca.is_absolute():
    ssl_ca = ROOT / ssl_ca          # 상대 경로는 프로젝트 루트 기준으로 해석
if not ssl_ca.exists():
    raise FileNotFoundError(f'RDS 인증서 없음: {ssl_ca}')

# ── 3) 엔진 설정 ─────────────────────────────────────────────
# 'mysql+pymysql' = MySQL에 pymysql 드라이버로 접속
db_connection_url = URL.create(
    'mysql+pymysql',
    username=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
    host=os.environ['DB_HOST'],
    port=int(os.environ.get('DB_PORT', 3306)),   # 지정하지 않으면 MySQL 기본 포트 3306
    database=os.environ['DB_NAME'],
)
engine = create_engine(db_connection_url, connect_args={'ssl': {'ca': str(ssl_ca)}})

# ── 4) 데이터 불러오기 ───────────────────────────────────────
# integrate: 국가×연도별로 군사비·GDP·수입점유율·분쟁위험도를 합쳐 둔 통합 테이블 (한 행 = 한 국가의 한 해)
df_demand = pd.read_sql("SELECT * FROM integrate", con=engine)
print(df_demand.shape)   # (행 수, 열 수)

### 지표와 표기

`TIV_5Y_Share`는 최근 5년 무기 **수입**량(TIV)의 세계 점유율 → **수입점유율**로 표기

In [ ]:
# 분석에 쓰는 5개 지표 (DB 컬럼명)
cols = ['current_usd', 'gdp_calculated', 'TIV_5Y_Share', 'human_hazard_score', 'share_gdp']

# 컬럼명 → 그래프·표에 표시할 한글 이름
LABEL = {
    'current_usd': '군사비',             # 군사비 지출액 (경상 달러)
    'gdp_calculated': 'GDP',             # 국내총생산
    'TIV_5Y_Share': '수입점유율',        # 최근 5년 무기 수입량(TIV)의 세계 점유율(%)
    'human_hazard_score': '분쟁위험도',  # 0~10 점수, 높을수록 위험
    'share_gdp': '군사비/GDP',           # GDP 대비 군사비 비중 (0~1 비율)
}
YEAR = 2023   # 단면 분석 기준 연도 (2~5절에서 사용)

# (Country, Year) 중복 여부 — 같은 국가·연도가 여러 행이면 한 나라가 두 번 세어지므로 먼저 확인
# 참고: 현재 잡히는 중복은 국가명(Country)이 비어 있는 행들이며 지표도 전부 결측이라 분석에는 영향 없음
print('중복 행 수:', df_demand.duplicated(['Country', 'Year']).sum())

# 결측으로 빠지는 국가 확인
y = df_demand[df_demand.Year == YEAR]                 # 기준 연도 행만 추출
print(f'\n{YEAR}년 전체: {len(y)}개국, 지표별 결측')
print(y[cols].isna().sum().rename(LABEL).to_string())  # 지표별 결측 개수

# 5개 지표 중 하나라도 비어 있으면 제외 → d가 이후 모든 단면 분석의 대상
d = y.dropna(subset=cols).copy()
print(f'\n{YEAR}년 분석 대상: {len(d)}개국 (결측 제외 {len(y) - len(d)}개국)')

# 수입점유율만 비어서 빠진 국가: "수입 없음"인지 "데이터 없음"인지 확인 필요
# 조건: 수입점유율은 결측 AND 나머지 4개 지표는 모두 값이 있음
only_tiv = y[y.TIV_5Y_Share.isna() & y[[c for c in cols if c != 'TIV_5Y_Share']].notna().all(axis=1)]
print(f'수입점유율만 결측이라 빠진 국가 {len(only_tiv)}개: {", ".join(only_tiv.Country.head(15))}')

# 기초 통계량. 평균이 중앙값(50%)보다 훨씬 크면 오른쪽으로 길게 치우친 분포
d[cols].describe().round(2)

### 스케일 변환

원본 `log1p`는 쓰지 않음. 값이 작은 비율 변수(군사비/GDP, 수입점유율)에서는 `log1p(x) ≈ x`라서 변환 효과가 없기 때문

- 금액·비율 4개 지표 → `log10`. 0이 섞인 지표는 최소 양수값의 절반을 더한 뒤 `log10`
- 분쟁위험도 → 0~10 점수라 변환하지 않음

In [ ]:
# 로그 변환할 지표: 금액(군사비·GDP)과 비율(수입점유율·군사비/GDP)
# 분쟁위험도는 0~10으로 범위가 정해진 점수라 변환하지 않음
LOG_COLS = ['current_usd', 'gdp_calculated', 'TIV_5Y_Share', 'share_gdp']


def log_scale(s):
    """양수만 있으면 log10, 0이 섞여 있으면 (최소 양수값 / 2)를 더한 뒤 log10"""
    s = s.astype(float)
    if (s > 0).all():
        return np.log10(s)
    # log10(0)은 -inf라서 계산 불가 → 아주 작은 값을 더해 0도 그래프에 남김
    # 더하는 값을 "최소 양수값의 절반"으로 잡으면 0인 국가가 가장 작은 양수 국가보다 아래에 놓여 순서가 유지됨
    return np.log10(s + s[s > 0].min() / 2)


tf = d[cols].copy()                     # 변환된 지표 (이후 박스플롯·산점도·VIF에서 공통 사용)
tf[LOG_COLS] = tf[LOG_COLS].apply(log_scale)   # 지표(열)마다 따로 변환. 분쟁위험도 열은 원본 그대로

# 0이 있는 지표 확인 (0이 있으면 위의 보정이 적용된 것)
print('0 값 개수:', (d[LOG_COLS] == 0).sum().rename(LABEL).to_dict())

# 변환 후 통계량. log10 값이 1 차이 = 원래 값이 10배 차이
tf.rename(columns=LABEL).describe().round(2)

---
## 2. 박스플롯 — 이상치 탐지

어떤 지표가 극단값에 휘둘리는지 알아야 나중에 축 스케일과 정규화 방식을 정할 수 있음

원본으로 그리면 상자가 바닥에 붙은 선 하나로 뭉개지므로 로그 스케일로 그림

In [ ]:
# 지표 5개를 가로로 나란히. 지표마다 단위가 달라 y축은 각자 따로 사용
fig, axes = plt.subplots(1, 5, figsize=(14, 4))
for ax, c in zip(axes, cols):
    # tf = 로그 변환된 값. 상자 = 가운데 50%(Q1~Q3), 가운데 선 = 중앙값, 수염 밖의 점 = 이상치
    sns.boxplot(y=tf[c], ax=ax, color='#7FB3D5', width=0.5)
    ax.set_title(LABEL[c] + (' (log10)' if c in LOG_COLS else ''))   # 로그 변환한 지표만 제목에 표시
    ax.set_ylabel('')
plt.suptitle(f'{YEAR}년 지표별 분포')
plt.tight_layout()
plt.show()

In [ ]:
# IQR 기준 이상치 개수 — 원본 스케일과 로그 스케일(박스플롯과 동일)을 나란히 비교
def iqr_outliers(s):
    # IQR(사분위 범위) = Q3 - Q1. [Q1 - 1.5×IQR, Q3 + 1.5×IQR] 밖이면 이상치 (박스플롯의 수염 기준과 같음)
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)   # 국가별 True/False


rows = []
for c in cols:
    # raw: 원본 값 기준 이상치 여부 / logged: 로그 변환 값 기준 이상치 여부
    raw, logged = iqr_outliers(d[c]), iqr_outliers(tf[c])
    rows.append({
        '지표': LABEL[c],
        '원본 이상치': raw.sum(),       # True 개수 = 이상치 국가 수
        '로그 이상치': logged.sum(),
        '원본 기준 상위 3개국': ', '.join(d[raw].nlargest(3, c).Country),   # 이상치 중 값이 가장 큰 3개국
    })
pd.DataFrame(rows).set_index('지표')

**읽는 법**

- 원본 스케일에서 이상치가 많이 잡히는 것은 분포가 한쪽으로 길게 치우쳐 있기 때문 (실제 "이상한 값"이라기보다 스케일 문제)
- 로그 스케일에서도 남는 이상치가 진짜 극단값
- 분쟁위험도는 0~10 점수라 구조적으로 극단값이 생기기 어려움

→ 군사비·GDP·수입점유율은 로그 또는 순위 변환 후 사용

---
## 3. 상관계수 히트맵

**세 가지를 같이 보는 이유**

- **Pearson (원본)** — 직선 관계. 이상치에 아주 약함
- **Pearson (로그)** — 로그 변환 후의 직선 관계
- **Spearman** — 순위 일치도. 곡선·이상치에 강건 (단조 변환에는 값이 변하지 않음)

원본 Pearson과 Spearman이 크게 다르면 "관계는 강한데 직선이 아니다"

### 3-1. 정규성 검정

Pearson 상관계수의 유의성 검정은 두 변수가 정규분포를 따른다고 가정함 → 상관분석 전에 지표별로 정규성부터 확인

- **H0**: 해당 지표는 정규분포를 따른다
- **H1**: 정규분포를 따르지 않는다

Shapiro-Wilk 검정 사용. p-value가 유의수준(`ALPHA`)보다 작으면 H0 기각 = 정규분포가 아님

In [ ]:
# ── 분석 방침: 분석 전에 미리 고정 ──────────────────────────
# 결과를 본 뒤에 기준을 바꾸면 원하는 결론에 맞추게 되므로 검정 전에 정해 둠
ALPHA = 0.05   # 유의수준 (이후 모든 검정에서 공통 사용)

rows = []
for c in cols:
    # 지표마다 검정할 버전 목록: 원본은 항상, 로그 변환한 지표는 log10도 함께
    variants = [('원본', d[c])]
    if c in LOG_COLS:
        # tf = 1절에서 만든 log10 변환 값 (수입점유율은 0을 보정한 뒤 변환한 값)
        variants.append(('log10 (0 보정)' if c == 'TIV_5Y_Share' else 'log10', tf[c]))
    if c == 'TIV_5Y_Share':
        # 0인 국가는 보정해도 한 점에 몰려 분포를 왜곡하므로, 아예 뺀 경우도 따로 확인
        variants.append(('log10 (0 제외)', np.log10(d[c][d[c] > 0])))

    for tag, s in variants:
        W, p = stats.shapiro(s)   # W: 1에 가까울수록 정규분포에 가까움, p: 유의확률
        rows.append({
            '지표': LABEL[c], '변환': tag, 'n': len(s),
            '왜도': round(stats.skew(s), 2),        # 0이면 좌우 대칭, 양수면 오른쪽 꼬리가 김
            '첨도': round(stats.kurtosis(s), 2),    # 0이면 정규분포 수준, 양수면 꼬리가 두꺼움
            'Shapiro W': round(W, 3),
            'p-value': f'{p:.2e}',
            '정규성': '기각' if p < ALPHA else '정규성 만족',
        })

pd.DataFrame(rows)

In [ ]:
# Q-Q plot: 정규성 시각화
# 점들이 빨간 직선 위에 놓일수록 정규분포에 가까움. 양 끝이 직선에서 휘어지면 꼬리가 길다는 뜻
fig, axes = plt.subplots(2, 5, figsize=(16, 6.5))
for j, c in enumerate(cols):
    # 윗줄: 원본 값
    stats.probplot(d[c], dist='norm', plot=axes[0, j])
    axes[0, j].set_title(f'{LABEL[c]} (원본)', fontsize=10)
    # 아랫줄: log10 변환 값. 분쟁위험도는 변환하지 않으므로 빈 칸으로 둠
    if c in LOG_COLS:
        stats.probplot(tf[c], dist='norm', plot=axes[1, j])
        axes[1, j].set_title(f'{LABEL[c]} (log10)', fontsize=10)
    else:
        axes[1, j].axis('off')

for ax in axes.ravel():
    ax.set_xlabel(''); ax.set_ylabel('')
    if ax.get_lines():                        # 빈 칸에는 선이 없으므로 건너뜀
        ax.get_lines()[0].set_markersize(3)   # [0] = 데이터 점, [1] = 기준 직선
plt.suptitle('Q-Q plot')
plt.tight_layout()
plt.show()

In [ ]:
# Q-Q plot: 수입점유율만 따로 — 0을 어떻게 처리하느냐에 따라 정규성이 달라지는지 확인
tiv = d.TIV_5Y_Share
panels = [
    ('수입점유율 (원본)', tiv),
    ('수입점유율 (log10, 0 보정)', tf.TIV_5Y_Share),          # 0에 작은 값을 더한 뒤 로그
    ('수입점유율 (log10, 0 제외)', np.log10(tiv[tiv > 0])),   # 0인 국가를 뺀 뒤 로그
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (title, s) in zip(axes, panels):
    stats.probplot(s, dist='norm', plot=ax)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.get_lines()[0].set_markersize(3)
plt.suptitle('Q-Q plot')
plt.tight_layout()
plt.show()

**읽는 법**

- 원본 값에서 정규성이 기각된 지표는 원본 Pearson의 p-value를 그대로 믿을 수 없음
- 로그 변환 후 "정규성 만족"으로 바뀐 지표는 로그 값으로 Pearson을 쓸 수 있음
- 로그 변환 후에도 기각되는 지표가 있으면 → 정규성 가정이 필요 없는 **Spearman을 기본**으로 사용
- 수입점유율은 0인 국가를 보정한 경우와 제외한 경우의 결과가 다르면, 0인 국가들이 분포를 왜곡하고 있다는 뜻

In [ ]:
# 상관계수 행렬 3종
# - d  = 원본 값, tf = 로그 변환 값
# - Spearman은 순위만 쓰므로 로그 변환 여부와 무관하게 값이 같음 → 원본으로 한 번만 계산
corr = {
    'Pearson (원본)': d[cols].corr('pearson'),
    'Pearson (로그)': tf[cols].corr('pearson'),
    'Spearman (순위)': d[cols].corr('spearman'),
}

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, (title, m) in zip(axes, corr.items()):
    # vmin/vmax를 -1~1로 고정해야 세 그림의 색을 같은 기준으로 비교할 수 있음
    # annot=True: 칸마다 숫자 표시, fmt='.2f': 소수 둘째 자리까지
    sns.heatmap(m.rename(index=LABEL, columns=LABEL), annot=True, fmt='.2f', cmap='RdYlBu',
                vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
# 해석에 쓰는 주요 쌍의 수치
# 히트맵에서 눈으로 찾지 않아도 되도록, 아래 "확인할 것"에서 다루는 쌍만 뽑아 표로 정리
pairs = [('current_usd', 'TIV_5Y_Share'),      # ① 군사비 ~ 수입점유율
         ('current_usd', 'gdp_calculated'),    # ③ 군사비 ~ GDP
         ('share_gdp', 'current_usd'), ('share_gdp', 'gdp_calculated')]   # ③ 군사비/GDP가 규모와 따로 움직이는지
pairs += [('human_hazard_score', c) for c in cols if c != 'human_hazard_score']   # ② 분쟁위험도 ~ 나머지 전부

# 행 = 지표 쌍, 열 = 상관계수 종류
pd.DataFrame(
    {title: [m.loc[a, b] for a, b in pairs] for title, m in corr.items()},
    index=[f'{LABEL[a]} ~ {LABEL[b]}' for a, b in pairs],
).round(2)

**확인할 것**

**① 군사비 ~ 수입점유율: 원본 Pearson과 Spearman의 차이**

원본 Pearson만 보면 "약한 상관"으로 읽혀 *군사비 큰 나라라고 무기를 많이 사는 건 아니다* 라는
틀린 결론이 나올 수 있음. Spearman이 훨씬 높다면 관계는 강하고, 형태가 직선이 아니라 곡선

**② 분쟁위험도 ~ 나머지 지표**

상관이 전부 낮다면 분쟁위험도는 다른 지표가 주지 못하는 독립적인 정보

**③ 군사비 ~ GDP**

상관이 매우 높다면 둘 다 "국가 규모"를 나타내는 사실상 같은 지표.
수입점유율도 규모와 함께 움직이는지, 군사비/GDP는 규모가 아니라 "부담 수준"이라 따로 움직이는지 확인

### 3-2. 상관계수 유의성 검정 + 다중비교 보정

위 히트맵의 상관계수는 "얼마나 닮았나"만 알려주고, 그 값이 우연히 나온 것인지는 알려주지 않음.
몇 나라를 조사했는지(n)까지 고려해 **그 상관이 통계적으로 믿을 만한지** 검정

- **H0**: ρ = 0 (두 지표는 관계 없음)
- **H1**: ρ ≠ 0 (관계 있음)

**다중비교 보정이 필요한 이유**

지표 5개 → 10쌍을 한꺼번에 검정. 관계가 전혀 없어도 그중 하나가 우연히 유의하게 나올 확률이
1 − 0.95¹⁰ ≈ 40%. 그래서 Benjamini-Hochberg FDR 보정으로 기준을 조임. 보정된 p-value를 **q**로 표기

In [ ]:
# ══════════════════════════════════════════════════════════════
# 상관계수 유의성 + 다중비교 보정
#
# [통계량] t = r × sqrt(n-2) / sqrt(1-r²),  자유도 n-2
#   같은 r = 0.2라도 n = 40이면 유의하지 않고 n = 150이면 유의함 → n을 항상 함께 표시
# ══════════════════════════════════════════════════════════════

def corr_ci(r, n, alpha=ALPHA, kind='spearman'):
    """상관계수의 신뢰구간 (Fisher z 변환).

    직접 구현하는 이유:
      scipy의 pearsonr(...).confidence_interval()은 Pearson 전용이고 spearmanr에는 없음.

    z 변환을 하는 이유:
      r의 표본분포는 비대칭. r이 1에 가까울수록 위쪽이 눌려서 '평균 ± 오차' 형태로 구간을 잡을 수 없음.
      z = arctanh(r)로 바꾸면 근사적으로 정규분포가 되어 구간을 잡을 수 있음.

    1.06은 무엇인가:
      Spearman은 값을 순위로 바꾸는 과정에서 정보가 조금 손실되어 분산이 Pearson보다 큼.
      z의 분산이 1/(n-3)이 아니라 1.06/(n-3) (Fieller, Hartley & Pearson 1957).
      이 보정 없이 Pearson 공식을 쓰면 구간이 실제보다 좁게 나와 "유의하다" 판정이 과하게 나옴.
    """
    if n < 5 or abs(r) >= 1:
        return (np.nan, np.nan)   # 표본이 너무 적거나 r = ±1이면 계산 불가
    z = np.arctanh(r)
    var = 1 / (n - 3) if kind == 'pearson' else 1.06 / (n - 3)   # z의 분산
    zc = stats.norm.ppf(1 - alpha / 2)                           # 95% 구간이면 약 1.96
    lo, hi = np.tanh([z - zc * np.sqrt(var), z + zc * np.sqrt(var)])   # z 구간을 다시 r 단위로 되돌림
    return (float(lo), float(hi))


rows = []
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):        # 모든 지표 쌍을 한 번씩 (5개 → 10쌍)
        a, b = cols[i], cols[j]
        s = d[[a, b]].dropna()
        n = len(s)
        rp, pp = stats.pearsonr(s[a], s[b])     # 직선 관계: rp = 상관계수, pp = p-value (원본 값 기준)
        rs, ps = stats.spearmanr(s[a], s[b])    # 순위(단조) 관계: rs = 상관계수, ps = p-value
        rows.append({
            'x': LABEL[a], 'y': LABEL[b], 'n': n,
            'Pearson r': round(rp, 3), '_pp': pp,
            'Spearman r': round(rs, 3), '_ps': ps,
            'Spearman 95%CI': '[{:.2f}, {:.2f}]'.format(*corr_ci(rs, n)),
            # 이 괴리가 0.2를 넘으면 '단조 관계는 강한데 직선이 아니다'는 신호
            '괴리(|rs|-|rp|)': round(abs(rs) - abs(rp), 3),
        })

t = pd.DataFrame(rows)

# FDR 보정: 10개의 p-value를 한꺼번에 넣어 보정된 값(q)을 받음
# multipletests 반환값 = (기각 여부, 보정된 p-value, ...)
_, q_p, _, _ = multipletests(t['_pp'], alpha=ALPHA, method='fdr_bh')
rej_s, q_s, _, _ = multipletests(t['_ps'], alpha=ALPHA, method='fdr_bh')
t['Pearson q'] = [f'{v:.2e}' for v in q_p]             # 보정된 Pearson p-value
t['Spearman p'] = t['_ps'].map(lambda v: f'{v:.2e}')   # 보정 전 Spearman p-value
t['Spearman q'] = [f'{v:.2e}' for v in q_s]            # 보정된 Spearman p-value
t['유의(FDR)'] = np.where(rej_s, 'O', '')              # Spearman q < ALPHA 인 쌍에 O

# 계산용 임시 열(_pp, _ps)은 빼고 표시
t.drop(columns=['_pp', '_ps'])

In [ ]:
# 별표 붙은 히트맵: 관계가 유의한 쌍에만 * 표시
def star(q):
    return '***' if q < .001 else '**' if q < .01 else '*' if q < .05 else ''


def annotated_heatmap(df, cols, method='spearman', ax=None):
    R = df[cols].corr(method)                                       # 상관계수 행렬
    P = pd.DataFrame(np.ones_like(R), index=cols, columns=cols)     # q값 행렬 (1로 채워 두고 아래에서 덮어씀)

    # 쌍마다 p-value 계산
    pairs = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            s = df[[cols[i], cols[j]]].dropna()
            fn = stats.pearsonr if method == 'pearson' else stats.spearmanr
            pairs.append((i, j, fn(s[cols[i]], s[cols[j]])[1]))

    # 여기서도 FDR 보정된 q값으로 별을 찍음 (보정 전 p로 찍으면 별이 과하게 붙음)
    _, q, _, _ = multipletests([p for *_, p in pairs], method='fdr_bh')
    for (i, j, _), qq in zip(pairs, q):
        P.iloc[i, j] = P.iloc[j, i] = qq      # 대칭이므로 양쪽 칸에 같은 값

    # 칸에 표시할 글자 = 상관계수(소수 둘째 자리) + 별표
    ann = R.round(2).astype(str) + P.apply(lambda col: col.map(star))
    np.fill_diagonal(ann.values, '1.00')      # 대각선(자기 자신)은 별 없이 1.00

    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 6))
    sns.heatmap(R, annot=ann, fmt='', cmap='RdYlBu', vmin=-1, vmax=1,   # fmt='': ann이 이미 문자열이라 서식 없음
                square=True, linewidths=.5, ax=ax,
                xticklabels=[LABEL[c] for c in cols],
                yticklabels=[LABEL[c] for c in cols])
    ax.set_title(f'{method.capitalize()} 상관 (FDR 보정, n={len(df)})\n'
                 f'*** q<.001   ** q<.01   * q<.05')
    return ax


fig, axes = plt.subplots(1, 2, figsize=(15, 6))
annotated_heatmap(d, cols, 'pearson', axes[0])    # 원본 값 기준 Pearson
annotated_heatmap(d, cols, 'spearman', axes[1])
plt.tight_layout()
plt.show()

**읽는 법**

- **Spearman q**가 `ALPHA`(0.05) 미만인 쌍만 "관계가 있다"고 서술할 수 있음
- **95% CI**가 0을 포함하면 계수가 커 보여도 방향(양/음)조차 확정할 수 없음
- **괴리**가 0.2를 넘으면 곡선 관계 의심 → 4절 산점도와 로그-로그 기울기로 형태 확인
- Pearson 열은 원본 값 기준. 3-1에서 원본의 정규성이 기각됐다면 Pearson의 유의성은 참고용으로만 볼 것

---
## 4. 산점도

상관계수는 관계를 숫자 하나로 압축

같은 상관계수라도 *고르게 퍼진 경우 / 두 덩어리인 경우 / 이상치가 만든 경우*가 전부 다름

따라서 산점도로 경향 확인

In [ ]:
# 전체 쌍 한눈에 (분쟁위험도 외에는 log10)
# 축 이름을 한글로 바꾸고, 로그 변환한 지표에는 '(log10)'을 붙임
axis_label = {c: LABEL[c] + (' (log10)' if c in LOG_COLS else '') for c in cols}
# corner=True: 대각선 아래 삼각형만 그림 (위쪽은 같은 그림의 반복이라 생략)
# 대각선 = 각 지표의 분포(히스토그램), 나머지 = 두 지표의 산점도
sns.pairplot(tf.rename(columns=axis_label), corner=True, plot_kws={'s': 20, 'alpha': 0.6})
plt.suptitle('지표 간 산점도 행렬', y=1.01)
plt.show()

In [ ]:
# 핵심 조합: 군사비 × 수입점유율, 색은 분쟁위험도
# 값은 원본(d)을 그대로 넣고 축만 로그로 바꿈 → 눈금이 실제 금액·점유율로 읽힘
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(d.current_usd, d.TIV_5Y_Share, c=d.human_hazard_score, cmap='OrRd',   # 색이 진할수록 분쟁위험도 높음
                s=60, alpha=0.8, edgecolor='k', linewidth=0.3)

ax.set_xscale('log')
ax.set_yscale('symlog', linthresh=0.01)   # 점유율 0인 국가가 있어 symlog (0.01 미만 구간은 선형, 그 위는 로그)
ax.set_xlabel('군사비 (log)')
ax.set_ylabel('수입점유율 % (symlog)')
ax.set_title('군사비 × 수입점유율')
plt.colorbar(sc, label='분쟁위험도')
plt.tight_layout()
plt.show()

In [ ]:
# 로그-로그 기울기: 군사비가 10배일 때 수입점유율이 몇 배인지 (수입점유율 0인 국가 제외)
pos = d[d.TIV_5Y_Share > 0]   # log10(0)은 계산할 수 없어 0인 국가는 제외
# log10(수입점유율) = 기울기 × log10(군사비) + 절편 의 1차 직선을 맞춤. polyfit 결과의 [0]이 기울기
slope = np.polyfit(np.log10(pos.current_usd), np.log10(pos.TIV_5Y_Share), 1)[0]
# 기울기 1 = 정비례(군사비 10배 → 점유율 10배), 1보다 작으면 군사비만큼은 늘지 않음. 10 ** 기울기 = 실제 배수
print(f'로그-로그 기울기 {slope:.2f} → 군사비 10배일 때 수입점유율 약 {10 ** slope:.1f}배 ({len(pos)}개국)')

**확인할 것**

- 점들이 왼쪽 아래에서 오른쪽 위로 이어지는지, 그 모양이 직선인지 완만한 곡선인지
- 위 셀의 기울기가 1보다 작으면 군사비가 늘어나는 만큼 수입점유율이 비례해서 늘지는 않는다는 뜻
- 색(분쟁위험도)이 특정 구간에 몰려 있는지, 전 구간에 섞여 있는지

---
## 5. 다중공선성 (VIF)

여러 지표를 합쳐 하나의 값으로 예측할 때 변수가 겹쳐서 기울기에 혼란을 주는 경우를 막기 위함

해당 변수가 다른 변수들에 의해 얼마나 설명되는지를 뜻함

기준은 5 이상 주의, 10 이상 심각

**수입점유율은 예측하려는 대상에 가까우므로 VIF 계산에서 제외** (설명변수끼리만 계산)

In [ ]:
# 구조적 종속 확인: 군사비/GDP = 군사비 ÷ GDP 라면, 로그에서는 log(비중) = log(군사비) - log(GDP)
# tf는 이미 log10 변환된 값이라 뺄셈만 하면 됨
implied = tf.current_usd - tf.gdp_calculated          # 두 지표로 만든 log(군사비/GDP)
# 직접 계산한 값과 DB의 군사비/GDP가 얼마나 일치하는지 상관계수로 확인
print(f'log(군사비) - log(GDP) 와 log(군사비/GDP) 의 상관: {implied.corr(tf.share_gdp):.4f}')
print('→ 1에 가까우면 세 지표 중 하나는 나머지 둘로 완전히 설명됨 (셋을 동시에 넣으면 안 됨)')

In [ ]:
def vif(frame):
    """상관행렬의 역행렬 대각원소 = VIF. 완전 종속이면 inf"""
    # VIF_i = 1 / (1 - R_i²), R_i² = i번째 변수를 나머지 변수들로 회귀했을 때의 설명력
    # 이 값은 상관행렬 역행렬의 i번째 대각원소와 같아서 회귀를 따로 돌리지 않아도 됨
    try:
        v = np.diag(np.linalg.inv(frame.corr().to_numpy()))
    except np.linalg.LinAlgError:
        v = np.full(frame.shape[1], np.inf)   # 역행렬이 없음 = 변수들이 완전히 종속
    return pd.Series(v, index=frame.columns)


# 설명변수 후보 4개 (수입점유율은 예측 대상에 가까워 제외)
predictors = ['current_usd', 'gdp_calculated', 'share_gdp', 'human_hazard_score']

# 비교할 조합: 전부 넣은 경우 + 서로 종속인 세 지표(군사비·GDP·군사비/GDP) 중 하나씩 뺀 경우
candidates = {
    '4개 전부': predictors,
    'GDP 제외': ['current_usd', 'share_gdp', 'human_hazard_score'],
    '군사비 제외': ['gdp_calculated', 'share_gdp', 'human_hazard_score'],
    '군사비/GDP 제외': ['current_usd', 'gdp_calculated', 'human_hazard_score'],
}

# 행 = 지표, 열 = 조합. 그 조합에 들어가지 않은 지표는 NaN으로 표시됨
# 로그 변환 값(tf) 기준으로 계산
pd.DataFrame({name: vif(tf[c]) for name, c in candidates.items()}).reindex(predictors).rename(index=LABEL).round(2)

**읽는 법**

- **4개 전부** — 군사비·GDP·군사비/GDP는 한 묶음으로 중복이라 VIF가 매우 크게 나옴 (원본 노트북의 15 / 13은 `log1p` 때문에 이 종속이 가려진 값)
- **하나를 뺀 조합** — 남은 지표의 VIF가 모두 5 미만인 조합이 사용 가능한 조합
- 군사비와 GDP만 남긴 조합은 둘의 상관이 높아 여전히 VIF가 클 수 있음

수출 유망국 판단에는 군사비(규모) + 군사비/GDP(부담 수준) + 분쟁위험도 조합이 직접적.
위 표에서 **GDP 제외** 열의 VIF가 낮게 나오는지 확인 후 결정